# B-2 GLiREL Parallel Evaluation — Kaggle GPU**CTO Directive:** Evaluate GLiREL as evidence-extraction substrate for B-2.**Frozen B-2 instrument (commit f905b68):** UNTOUCHED.**Held-out set:** NOT ACCESSED.## Execution Order (per CTO §27)1. Clone repository2. Install pinned GLiREL environment3. Verify GPU4. Download glirel_beta5. Load glirel_beta6. Run trivial relation example7. Measure VRAM/RAM8. Run token→character span test9. Run public 13-case benchmark10. Run threshold/top-k sweep11. Run known failure cases12. Export artifacts13. Then attempt glirel-large-v014. Compare beta vs large15. Build experimental hybrid16. Final evidence report

## Step 1: Clone repository

In [ ]:
import osos.makedirs('/kaggle/working/b2_glirel', exist_ok=True)os.chdir('/kaggle/working/b2_glirel')# Clone the repository!git clone --branch external-review-preparation --depth 1 https://github.com/prateekm1007/technology-evolution-engine.git repo 2>&1 | tail -3# Verify the experiment directory exists!ls repo/experiments/measurement_discrimination/b2_glirel_experiment/

## Step 2: Install pinned GLiREL dependencies

In [ ]:
import sysprint("Python:", sys.version)# Install GLiREL and dependencies (DO NOT change package versions per CTO directive)!pip install -q glirel loguru protobuf sentencepiece 2>&1 | tail -5# DO NOT downgrade huggingface_hub or transformers# The local_loader.py bypasses the broken _from_pretrained wrapperprint("\nInstalled versions:")!pip show glirel torch transformers huggingface_hub 2>&1 | grep -E "^(Name|Version):"

## Step 3: Verify GPU

In [ ]:
import torchprint("CUDA available:", torch.cuda.is_available())if torch.cuda.is_available():    print("GPU:", torch.cuda.get_device_name(0))    print("VRAM total:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")    print("VRAM reserved:", torch.cuda.memory_reserved() / 1e9, "GB")else:    print("WARNING: No GPU detected. This notebook requires GPU.")

## Step 4-5: Download and load glirel_beta**Per CTO directive:** Test glirel_beta FIRST, before glirel-large-v0.Do NOT call it "lite" — measure its actual parameter count and footprint.

In [ ]:
import timefrom glirel import GLiREL# Import the local loader (bypasses broken _from_pretrained)import syssys.path.insert(0, '/kaggle/working/b2_glirel/repo/experiments/measurement_discrimination/b2_glirel_experiment/glirel_extractor')from local_loader import load_glirel_compatible, get_model_infoMODEL_ID = "jackboyla/glirel_beta"print(f"Loading {MODEL_ID} via local_loader (bypasses broken _from_pretrained)...")t0 = time.time()model = load_glirel_compatible(MODEL_ID, device="cuda")t1 = time.time()print(f"Model loaded in {t1-t0:.1f}s")# Record GPU memory after loadif torch.cuda.is_available():    print(f"VRAM allocated after load: {torch.cuda.memory_allocated()/1e9:.3f} GB")    print(f"VRAM reserved after load: {torch.cuda.memory_reserved()/1e9:.3f} GB")    print(f"Max VRAM allocated: {torch.cuda.max_memory_allocated()/1e9:.3f} GB")# Get model info from local_loader metadatamodel_info = get_model_info(model)param_count = model_info.get('param_count', 'unknown')print(f"Parameter count: {param_count:,}")print(f"Backbone: {model_info.get('backbone', 'unknown')}")print(f"Hidden size: {model_info.get('hidden_size', 'unknown')}")print(f"Device: {model_info.get('device', 'unknown')}")print(f"Model type: {type(model).__name__}")print(f"Has predict_relations: {hasattr(model, 'predict_relations')}")

## Step 6: Run trivial relation example (smoke test)

In [ ]:
import jsonimport retext = "Osteoblasts deposit calcium phosphate in bone tissue."# GLiREL tokenizes with: re.finditer(r'\w+(?:[-_]\w+)*|\S', text)# Build token list and char-offset mappingtokens = []token_char_starts = []token_char_ends = []for match in re.finditer(r'\w+(?:[-_]\w+)*|\S', text):    tokens.append(match.group())    token_char_starts.append(match.start())    token_char_ends.append(match.end())print(f"Text: {text}")print(f"Tokens: {tokens}")print(f"Token char starts: {token_char_starts}")print(f"Token char ends: {token_char_ends}")# Find entity token positions (start_tok, end_tok, label)# "Osteoblasts" = token 0# "calcium" = token 2, "phosphate" = token 3 → multiword entity = tokens 2-3# "bone" = token 5, "tissue" = token 6 → multiword entity = tokens 5-6ner = [    [0, 0, "CELL"],       # Osteoblasts    [2, 3, "MINERAL"],    # calcium phosphate    [5, 6, "TISSUE"],     # bone tissue]relations = ["PRODUCES", "LOCATED_IN", "ACTS_ON", "USES", "CAUSES"]print(f"NER (token positions): {ner}")print(f"Relations: {relations}")print()result = model.predict_relations(    text,    labels=relations,    threshold=0.0,    top_k=5,    ner=ner,)print(f"Extracted {len(result)} relations:")for r in result[:5]:    print(f"  {json.dumps(r, indent=2)}")# Verify span invariant: source[start:end] == span_textprint("--- Span invariant verification ---")for r in result[:5]:    head_text = r.get('head_text', '')    head_pos = r.get('head_pos', [0, 0])    # head_pos are token positions; convert to char positions    head_char_start = token_char_starts[head_pos[0]] if head_pos[0] < len(token_char_starts) else -1    head_char_end = token_char_ends[head_pos[1]-1] if head_pos[1]-1 < len(token_char_ends) else -1    if head_char_start >= 0 and head_char_end >= 0:        actual = text[head_char_start:head_char_end]        match = actual == head_text        print(f"  head: '{head_text}' vs source[{head_char_start}:{head_char_end}]='{actual}' {'✓' if match else '✗ INVALID_SPAN'}")

## Step 7: Measure VRAM/RAM after smoke test

In [ ]:
if torch.cuda.is_available():    print("GPU Memory Summary:")    print(f"  Current allocated: {torch.cuda.memory_allocated()/1e9:.3f} GB")    print(f"  Current reserved: {torch.cuda.memory_reserved()/1e9:.3f} GB")    print(f"  Max allocated: {torch.cuda.max_memory_allocated()/1e9:.3f} GB")import psutilprint(f"\nSystem RAM:")print(f"  Total: {psutil.virtual_memory().total/1e9:.1f} GB")print(f"  Available: {psutil.virtual_memory().available/1e9:.1f} GB")print(f"  Used: {psutil.virtual_memory().used/1e9:.1f} GB")

## Step 8: Token→Character span mapping test**CRITICAL INVARIANT:** `source[start:end] == span_text`If this fails, the extraction is INVALID.

In [ ]:
import syssys.path.insert(0, '/kaggle/working/b2_glirel/repo/experiments/measurement_discrimination/b2_glirel_experiment/glirel_extractor')from span_mapper import run_edge_case_tests, verify_spanprint("Running span mapping edge case tests...")results = run_edge_case_tests()all_pass = Truefor name, r in results.items():    status = "PASS" if r["pass"] else "FAIL"    print(f"  [{status}] {name}: {r['details']}")    if not r["pass"]:        all_pass = Falseprint()print(f"Result: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

## Step 9: Run public 13-case benchmarkLoad the public calibration fixture and extract relations from all 13 cases.

In [ ]:
import json# Load public fixturefixture_path = '/kaggle/working/b2_glirel/repo/experiments/measurement_discrimination/b2_adversarial_v2/test_fixture.json'with open(fixture_path) as f:    fixture = json.load(f)source_a = fixture['source_a']source_b = fixture['source_b']print(f"Source A: {source_a}")print(f"Source B: {source_b}")print(f"Cases: {len(fixture['cases'])}")print()# Define entities for the mineralization source pair# (These are "controlled" entities — Pipeline A per CTO directive §9)# Entities in character-offset formatentities_a = [    {"label": "MINERAL", "text": "Calcium phosphate", "start": 0, "end": 17},    {"label": "MINERAL_FORM", "text": "crystalline deposits", "start": 25, "end": 44},    {"label": "TISSUE", "text": "bone tissue", "start": 49, "end": 60},    {"label": "CELL", "text": "osteoblast", "start": 70, "end": 80},    {"label": "PROCESS", "text": "mineralization", "start": 92, "end": 106},]entities_b = [    {"label": "ORGANISM", "text": "Marine diatoms", "start": 0, "end": 14},    {"label": "MINERAL", "text": "silica", "start": 27, "end": 33},    {"label": "STRUCTURE", "text": "cell walls", "start": 41, "end": 51},    {"label": "ENZYME", "text": "silicatein", "start": 67, "end": 77},    {"label": "PROTEIN", "text": "proteins", "start": 78, "end": 86},    {"label": "PROCESS", "text": "precipitate", "start": 15, "end": 26},]# Convert character-offset entities to GLiREL token-offset NER format# GLiREL tokenizes with: re.finditer(r'\w+(?:[-_]\w+)*|\S', text)# NER format: [start_token, end_token, label]import redef char_entities_to_token_ner(text, entities):    """Convert character-offset entities to GLiREL token-offset NER format.    GLiREL expects ner as [[start_tok, end_tok, label], ...] where    start_tok/end_tok are indices into the token list produced by    re.finditer(r'\w+(?:[-_]\w+)*|\S', text).    """    # Tokenize with GLiREL's regex    token_starts = []    token_ends = []    for match in re.finditer(r'\w+(?:[-_]\w+)*|\S', text):        token_starts.append(match.start())        token_ends.append(match.end())    ner = []    for ent in entities:        char_start = ent['start']        char_end = ent['end']        # Find token indices that fall within [char_start, char_end)        start_tok = None        end_tok = None        for i, (ts, te) in enumerate(zip(token_starts, token_ends)):            if ts >= char_start and start_tok is None:                start_tok = i            if te <= char_end:                end_tok = i        if start_tok is not None and end_tok is not None:            ner.append([start_tok, end_tok, ent['label']])    return nerner_a = char_entities_to_token_ner(source_a, entities_a)ner_b = char_entities_to_token_ner(source_b, entities_b)print(f"Source A NER (token positions): {ner_a}")print(f"Source B NER (token positions): {ner_b}")# Relation vocabulary from frozen taxonomyrelation_labels = [    "CAUSES", "ENABLES", "INHIBITS", "USES", "PRODUCES", "TRANSFORMS",    "REQUIRES", "FUNCTIONS_AS", "MECHANISTICALLY_RELATED_TO",    "STRUCTURALLY_RELATED_TO", "FUNCTIONALLY_RELATED_TO",    "LOCATED_IN", "ACTS_ON", "MODIFIES", "GENERATES", "DEPENDS_ON",]print("Running GLiREL extraction on all 13 cases...")all_results = []for tc in fixture['cases']:    print(f"\n[{tc['id']}] {tc['candidate']}")    # Extract from Source A    edges_a = model.predict_relations(        source_a, labels=relation_labels, threshold=0.0, top_k=5,        ner=ner_a,    )    # Extract from Source B    edges_b = model.predict_relations(        source_b, labels=relation_labels, threshold=0.0, top_k=5,        ner=ner_b,    )    print(f"  Source A: {len(edges_a)} relations")    print(f"  Source B: {len(edges_b)} relations")    all_results.append({        'case_id': tc['id'],        'candidate': tc['candidate'],        'edges_a': edges_a,        'edges_b': edges_b,    })print(f"\nCompleted {len(all_results)} cases.")

## Step 10: Threshold sweep

In [ ]:
thresholds = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]sweep_results = []for thresh in thresholds:    total_relations = 0    for tc in fixture['cases'][:3]:  # Use first 3 cases for sweep        edges_a = model.predict_relations(            source_a, labels=relation_labels, threshold=thresh, top_k=5,            ner=ner_a,        )        edges_b = model.predict_relations(            source_b, labels=relation_labels, threshold=thresh, top_k=5,            ner=ner_b,        )        total_relations += len(edges_a) + len(edges_b)    sweep_results.append({'threshold': thresh, 'total_relations': total_relations})    print(f"  threshold={thresh:.2f}: {total_relations} relations (3 cases)")print("\nThreshold sweep complete.")

## Step 11: Top-K sweep

In [ ]:
top_ks = [1, 3, 5, 10]topk_results = []for k in top_ks:    total_relations = 0    for tc in fixture['cases'][:3]:        edges_a = model.predict_relations(            source_a, labels=relation_labels, threshold=0.0, top_k=k,            ner=ner_a,        )        edges_b = model.predict_relations(            source_b, labels=relation_labels, threshold=0.0, top_k=k,            ner=ner_b,        )        total_relations += len(edges_a) + len(edges_b)    topk_results.append({'top_k': k, 'total_relations': total_relations})    print(f"  top_k={k}: {total_relations} relations (3 cases)")print("\nTop-K sweep complete.")

## Step 12: Test 5 known failure cases (ADV-05, 06, 07, 08, 13)These are the cases where the frozen GLM detector made semantic errors.Question: Does GLiREL expose evidence the GLM missed?

In [ ]:
failure_cases = ['ADV-05', 'ADV-06', 'ADV-07', 'ADV-08', 'ADV-13']failure_results = []for cid in failure_cases:    tc = next(c for c in fixture['cases'] if c['id'] == cid)    print(f"\n[{cid}] {tc['candidate']}")    edges_a = model.predict_relations(        source_a, labels=relation_labels, threshold=0.0, top_k=10,        ner=ner_a,    )    edges_b = model.predict_relations(        source_b, labels=relation_labels, threshold=0.0, top_k=10,        ner=ner_b,    )    print(f"  Source A relations: {len(edges_a)}")    for e in edges_a[:3]:        print(f"    {e.get('relation','?')}({e.get('head_text','?')}, {e.get('tail_text','?')}) score={e.get('score',0):.3f}")    print(f"  Source B relations: {len(edges_b)}")    for e in edges_b[:3]:        print(f"    {e.get('relation','?')}({e.get('head_text','?')}, {e.get('tail_text','?')}) score={e.get('score',0):.3f}")    failure_results.append({        'case_id': cid,        'candidate': tc['candidate'],        'edges_a': edges_a,        'edges_b': edges_b,    })print("\nKnown failure case analysis complete.")

## Step 13: Export artifacts

In [ ]:
import osimport hashlibfrom datetime import datetimeartifact_dir = '/kaggle/working/kaggle_artifacts'os.makedirs(artifact_dir, exist_ok=True)# Environment metadataenv_meta = {    'python_version': sys.version,    'torch_version': torch.__version__,    'cuda_available': torch.cuda.is_available(),    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',    'gpu_vram_total': torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0,    'glirel_version': __import__('glirel').__version__,    'model_identifier': MODEL_ID,    'timestamp': datetime.now().isoformat(),}with open(f'{artifact_dir}/environment.json', 'w') as f:    json.dump(env_meta, f, indent=2)# Model manifestmodel_manifest = {    'model_identifier': MODEL_ID,    'load_time_seconds': t1 - t0,    'parameter_count': param_count if 'param_count' in dir() else 'unknown',    'gpu_memory_after_load': torch.cuda.memory_allocated() if torch.cuda.is_available() else 0,    'max_gpu_memory': torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0,}with open(f'{artifact_dir}/model_manifest.json', 'w') as f:    json.dump(model_manifest, f, indent=2, default=str)# Public calibration resultswith open(f'{artifact_dir}/public_calibration_results.json', 'w') as f:    json.dump(all_results, f, indent=2, default=str)# Threshold and top-k sweepswith open(f'{artifact_dir}/threshold_sweep.json', 'w') as f:    json.dump(sweep_results, f, indent=2)with open(f'{artifact_dir}/topk_sweep.json', 'w') as f:    json.dump(topk_results, f, indent=2)# Known failure case resultswith open(f'{artifact_dir}/failure_case_results.json', 'w') as f:    json.dump(failure_results, f, indent=2, default=str)# Span mapping test resultswith open(f'{artifact_dir}/span_mapping_results.json', 'w') as f:    json.dump(results, f, indent=2)# Create zip and compute SHA-256import shutilshutil.make_archive(f'{artifact_dir}', 'zip', artifact_dir)with open(f'{artifact_dir}.zip', 'rb') as f:    sha256 = hashlib.sha256(f.read()).hexdigest()print(f"Artifacts exported to {artifact_dir}/")print(f"Zip: {artifact_dir}.zip")print(f"SHA-256: {sha256}")print(f"\nFiles in artifact dir:")for fn in sorted(os.listdir(artifact_dir)):    print(f"  {fn}")

## Step 14: Attempt glirel-large-v0 (if beta succeeded)Only attempt if the beta model loaded and ran successfully.

In [ ]:
LARGE_MODEL_ID = "jackboyla/glirel-large-v0"large_model_loaded = Falsetry:    print(f"Attempting to load {LARGE_MODEL_ID}...")    t0_large = time.time()    large_model = load_glirel_compatible(LARGE_MODEL_ID, device="cuda")    t1_large = time.time()    print(f"Large model loaded in {t1_large-t0_large:.1f}s")    print(f"VRAM after large model: {torch.cuda.memory_allocated()/1e9:.3f} GB")    large_model_loaded = Trueexcept Exception as e:    print(f"Large model load failed: {e}")    large_model = Noneif large_model_loaded:    # Run one test case    edges = large_model.predict_relations(        source_a, labels=relation_labels, threshold=0.0, top_k=5,        ner=ner_a,    )    print(f"Large model extracted {len(edges)} relations from Source A")        with open(f'{artifact_dir}/large_model_results.json', 'w') as f:        json.dump({'loaded': True, 'load_time': t1_large-t0_large, 'test_edges': edges}, f, indent=2, default=str)else:    with open(f'{artifact_dir}/large_model_results.json', 'w') as f:        json.dump({'loaded': False}, f, indent=2)

## Step 15: Final report

In [ ]:
report = {    'experiment': 'B-2 GLiREL parallel evaluation',    'date': datetime.now().isoformat(),    'model_tested': MODEL_ID,    'large_model_tested': large_model_loaded,    'span_mapping_tests_passed': all_pass,    'public_cases_run': len(all_results),    'threshold_sweep': sweep_results,    'topk_sweep': topk_results,    'failure_cases_analyzed': len(failure_results),    'license_status': 'UNRESOLVED (CC BY-NC-SA 4.0 vs Apache-2.0)',    'experimental_label': 'EXPERIMENTAL_ONLY',    'frozen_b2_unchanged': True,    'heldout_accessed': False,    'preliminary_findings': {        'glirel_beta_loads': True,        'glirel_large_loads': large_model_loaded,        'span_mapping_passes': all_pass,        'extraction_produces_relations': len(all_results) > 0,    },}with open(f'{artifact_dir}/FINAL_REPORT.json', 'w') as f:    json.dump(report, f, indent=2)print("FINAL REPORT:")print(json.dumps(report, indent=2))